# 03 — Label Creation

**Purpose:** Compute binary performance labels from manually collected student response data.

**Prerequisites:**
- `data/raw/student_responses.csv` must exist (see `data_collection_manual.md` Part C)
- `data/external/answer_key.json` must be filled in (see Part D)

**Inputs:**
- `data/raw/student_responses.csv`
- `data/external/answer_key.json`
- `data/processed/features_per_task.parquet` (for participant ID cross-check)

**Output:**
- `data/processed/labels.csv`

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import (
    DATA_RAW, FEATURES_PER_TASK, LABELS_CSV, ANSWER_KEY_JSON,
    TASKS, PERFORMANCE_LABEL_COL, SPEED_LABEL_COL,
    LABEL_HIGH, LABEL_LOW, LABEL_FAST, LABEL_SLOW,
    MetricsCols
)

RESPONSES_CSV = DATA_RAW / 'student_responses.csv'

## 1. Load student responses

In [ ]:
if not RESPONSES_CSV.exists():
    raise FileNotFoundError(
        f"\n{RESPONSES_CSV} not found.\n"
        "Create it by following data_collection_manual.md Part C.\n"
        "Expected columns: participant_id, findDice_response, findYummy_response, "
        "frogInBathroom_response, headphoneInBathroom_response, frog_response, "
        "whoCheats_response, whoThief_response\n"
        "Valid values: A, B, C, D, or NA"
    )

responses = pd.read_csv(RESPONSES_CSV)
# Standardize participant ID to match Tobii (lowercase, stripped)
responses['participant_id'] = responses['participant_id'].astype(str).str.strip().str.lower()
print(f"Loaded {len(responses)} participant responses")
responses.head()

## 2. Load answer key

In [ ]:
with open(ANSWER_KEY_JSON) as f:
    answer_key = json.load(f)

answer_key = {k: v for k, v in answer_key.items() if not k.startswith('_')}

# Validate no '?' remain
unfilled = [t for t, v in answer_key.items() if v == '?']
if unfilled:
    raise ValueError(
        f"Answer key not complete. Fill in the correct answer for: {unfilled}\n"
        "See data_collection_manual.md Part D."
    )
print("Answer key:")
for task, answer in answer_key.items():
    print(f"  {task}: {answer}")

## 3. Compute per-task accuracy

In [ ]:
for task in TASKS:
    response_col = f'{task}_response'
    correct_col  = f'{task}_correct'

    if response_col not in responses.columns:
        print(f"WARNING: column '{response_col}' not found — setting to NaN")
        responses[correct_col] = np.nan
        continue

    correct_answer = answer_key.get(task)
    responses[correct_col] = (
        responses[response_col].str.strip().str.upper() == correct_answer.upper()
    ).astype(float)
    # NA responses → NaN
    na_mask = responses[response_col].str.strip().str.upper() == 'NA'
    responses.loc[na_mask, correct_col] = np.nan

correct_cols = [f'{t}_correct' for t in TASKS]
responses['total_score']   = responses[correct_cols].sum(axis=1)
responses['accuracy_rate'] = responses['total_score'] / len(TASKS)

print("Score distribution:")
responses[['total_score', 'accuracy_rate']].describe().round(3)

## 4. Create performance label (High / Low via median split)

In [ ]:
median_accuracy = responses['accuracy_rate'].median()
print(f"Median accuracy rate: {median_accuracy:.3f} ({median_accuracy * len(TASKS):.1f}/{len(TASKS)} correct)")

responses[PERFORMANCE_LABEL_COL] = np.where(
    responses['accuracy_rate'] >= median_accuracy,
    LABEL_HIGH,  # 1 = High-performing
    LABEL_LOW    # 0 = Low-performing
)

label_counts = responses[PERFORMANCE_LABEL_COL].value_counts()
print(f"\nLabel distribution:")
print(f"  High (1): {label_counts.get(LABEL_HIGH, 0)}")
print(f"  Low  (0): {label_counts.get(LABEL_LOW,  0)}")

imbalance_ratio = label_counts.min() / label_counts.max()
if imbalance_ratio < 0.43:  # worse than 30/70
    print(f"\nWARNING: Class imbalance ratio = {imbalance_ratio:.2f}. Consider using balanced metrics.")

## 5. Create speed label (Fast / Slow) from response time

In [ ]:
features = pd.read_parquet(FEATURES_PER_TASK)

# Aggregate mean response time (Time_to_first_mouse_click) across all tasks per participant
rt_col = f'correct_{MetricsCols.TIME_FIRST_MOUSE_CLICK}'

if rt_col in features.columns:
    mean_rt = (
        features.groupby('participant_id')[rt_col]
        .mean()
        .reset_index()
        .rename(columns={rt_col: 'mean_response_time_ms'})
    )
    responses = responses.merge(mean_rt, on='participant_id', how='left')

    median_rt = responses['mean_response_time_ms'].median()
    print(f"Median response time: {median_rt:.0f} ms")

    responses[SPEED_LABEL_COL] = np.where(
        responses['mean_response_time_ms'] <= median_rt,
        LABEL_FAST,  # 1 = Fast
        LABEL_SLOW   # 0 = Slow
    )
else:
    print(f"Column '{rt_col}' not found in features — speed_label set to NaN")
    responses[SPEED_LABEL_COL] = np.nan

print("\nSpeed label distribution:")
print(responses[SPEED_LABEL_COL].value_counts())

## 6. Participant ID cross-check

In [ ]:
features_participants = set(features['participant_id'].unique())
label_participants    = set(responses['participant_id'].unique())

only_in_labels   = label_participants - features_participants
only_in_features = features_participants - label_participants

print(f"Participants in responses only (no eye-tracking): {only_in_labels}")
print(f"Participants in eye-tracking only (no responses): {only_in_features}")

if only_in_labels:
    print("\nWARNING: Some response records have no matching eye-tracking data. Check participant IDs.")

## 7. Visualize score and label distributions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Accuracy histogram
axes[0].hist(responses['accuracy_rate'], bins=10, color='steelblue', edgecolor='white')
axes[0].axvline(median_accuracy, color='red', linestyle='--', label=f'Median={median_accuracy:.2f}')
axes[0].set_title('Accuracy Rate Distribution')
axes[0].set_xlabel('Accuracy rate')
axes[0].legend()

# Label bar chart
label_map = {LABEL_HIGH: 'High', LABEL_LOW: 'Low'}
label_display = responses[PERFORMANCE_LABEL_COL].map(label_map).value_counts()
label_display.plot(kind='bar', ax=axes[1], color=['#2ecc71', '#e74c3c'], edgecolor='white')
axes[1].set_title('Class Distribution')
axes[1].set_ylabel('Count')
axes[1].set_xlabel('')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.savefig('../outputs/figures/03_label_distribution.png', dpi=150)
plt.show()

## 8. Save labels

In [ ]:
output_cols = (['participant_id', 'total_score', 'accuracy_rate',
                PERFORMANCE_LABEL_COL, SPEED_LABEL_COL] +
               correct_cols)

# Add mean_response_time if it exists
if 'mean_response_time_ms' in responses.columns:
    output_cols.append('mean_response_time_ms')

labels_out = responses[[c for c in output_cols if c in responses.columns]]
labels_out.to_csv(LABELS_CSV, index=False)

print(f"Saved: {LABELS_CSV}")
print(f"Shape: {labels_out.shape}")
labels_out.head()